In [4]:
import json

# --- Setup ---
# This script reads from 'export.geojson' in the same directory
# -----------

file_name = 'export.geojson'

# Keep track of all unique tag keys and highway types we find
all_found_tags = set()
highway_types_found = set()
elements_processed = 0

# Limit detailed output to first N features
DETAIL_LIMIT = 10

print(f"Attempting to load data from '{file_name}'...")

try:
    with open(file_name, 'r', encoding='utf-8') as f:
        osm_data = json.load(f)
    
    # GeoJSON uses 'features' instead of 'elements'
    features = osm_data.get('features', [])
    
    if not features:
        print("Found 'export.geojson', but it seems to be empty or has no 'features' key.")
    else:
        print(f"Successfully loaded {len(features)} features.")
        print(f"Showing details for first {min(DETAIL_LIMIT, len(features))} features...\n")
        
        # Loop through each feature in the GeoJSON
        for i, feature in enumerate(features):
            elements_processed += 1
            
            tags = feature.get('properties', {})
            
            # Only show detailed output for first DETAIL_LIMIT features
            if i < DETAIL_LIMIT:
                # Extract ID from the 'id' field (format: "way/123456")
                feature_id = feature.get('id', 'Unknown')
                print(f"--- Element {i+1} (ID: {feature_id}) ---")
                
                if not tags:
                    print("  This feature has no properties.")
                else:
                    # --- 1. Main Highway Info ---
                    highway_type = tags.get('highway')
                    if highway_type:
                        print(f"  ➡️ Highway Type: {highway_type}")
                    else:
                        print("  ➡️ Highway Type: Not specified")

                    # --- 2. Key Details ---
                    print(f"  Name: {tags.get('name', 'N/A')}")
                    print(f"  Max Speed: {tags.get('maxspeed', 'N/A')}")
                    print(f"  Oneway: {tags.get('oneway', 'N/A (Assumed two-way)')}")
                    print(f"  Sidewalk: {tags.get('sidewalk', 'N/A')}")
                    print(f"  Cycleway: {tags.get('cycleway', 'N/A')}")
                    print(f"  Is Bridge: {tags.get('bridge', 'No')}")
                    print(f"  Is Tunnel: {tags.get('tunnel', 'No')}")
                    print(f"  Surface: {tags.get('surface', 'N/A')}")
                    print(f"  Lit: {tags.get('lit', 'N/A')}")

                    # --- 3. Geometry Info ---
                    geometry = feature.get('geometry', {})
                    coordinates = geometry.get('coordinates', [])
                    print(f"  Geometry Points: {len(coordinates)}")
                print()  # Blank line between features
            
            # Track all found tags for ALL features
            if tags:
                all_found_tags.update(tags.keys())
                highway_type = tags.get('highway')
                if highway_type:
                    highway_types_found.add(highway_type)

        # --- Final Summary ---
        print("\n================== 📊 PARSING SUMMARY ==================")
        print(f"Processed {elements_processed} features total.")
        
        print("\nDiscovered Highway Types:")
        if highway_types_found:
            for ht in sorted(highway_types_found):
                print(f"  - {ht}")
        else:
            print("  No 'highway' tags were found.")
            
        print("\nAll Discovered Tag Keys:")
        if all_found_tags:
            for key in sorted(all_found_tags):
                print(f"  - {key}")
        else:
            print("  No tags were found at all.")
        print("=========================================================")


except FileNotFoundError:
    print(f"❌ Error: File not found: '{file_name}'")
    print("Please create this file in the same directory and paste your JSON results into it.")
except json.JSONDecodeError as e:
    print(f"❌ Error: Could not decode JSON from '{file_name}'.")
    print(f"  Details: {e}")
    print("  Make sure you copied the *entire* JSON output correctly.")

Attempting to load data from 'export.geojson'...
Successfully loaded 1377 features.
Showing details for first 10 features...

--- Element 1 (ID: way/222748921) ---
  ➡️ Highway Type: pedestrian
  Name: N/A
  Max Speed: N/A
  Oneway: N/A (Assumed two-way)
  Sidewalk: N/A
  Cycleway: N/A
  Is Bridge: No
  Is Tunnel: No
  Surface: paving_stones
  Lit: yes
  Geometry Points: 1

--- Element 2 (ID: way/222748922) ---
  ➡️ Highway Type: pedestrian
  Name: N/A
  Max Speed: N/A
  Oneway: N/A (Assumed two-way)
  Sidewalk: N/A
  Cycleway: N/A
  Is Bridge: No
  Is Tunnel: No
  Surface: paving_stones
  Lit: yes
  Geometry Points: 1

--- Element 3 (ID: way/222748923) ---
  ➡️ Highway Type: pedestrian
  Name: N/A
  Max Speed: N/A
  Oneway: N/A (Assumed two-way)
  Sidewalk: N/A
  Cycleway: N/A
  Is Bridge: No
  Is Tunnel: No
  Surface: paving_stones
  Lit: yes
  Geometry Points: 1

--- Element 4 (ID: way/311988349) ---
  ➡️ Highway Type: platform
  Name: N/A
  Max Speed: N/A
  Oneway: N/A (Assumed two

In [3]:
import pandas as pd
import random

# --- Setup ---
file_path = 'data_processed/BRON_cleaned/ongevallen_2024_clean.csv'

print(f"Loading data from '{file_path}'...\n")

try:
    # Load the CSV file
    df = pd.read_csv(file_path)
    
    # Columns to analyze
    columns_to_analyze = ['junctie_id', 'wegvak_id', 'straatnaam']
    
    # Check if columns exist
    missing_cols = [col for col in columns_to_analyze if col not in df.columns]
    if missing_cols:
        print(f"⚠️ Warning: The following columns were not found in the dataset: {missing_cols}")
        columns_to_analyze = [col for col in columns_to_analyze if col in df.columns]
    
    if not columns_to_analyze:
        print("❌ None of the specified columns were found in the dataset.")
    else:
        total_rows = len(df)
        print(f"Total rows in dataset: {total_rows:,}\n")
        print("=" * 80)
        
        # Analyze each column
        for col in columns_to_analyze:
            print(f"\n📊 COLUMN: {col}")
            print("-" * 80)
            
            # Number of unique entries
            unique_count = df[col].nunique()
            unique_pct = (unique_count / total_rows) * 100
            
            # Missing values
            missing_count = df[col].isna().sum()
            missing_pct = (missing_count / total_rows) * 100
            
            # Non-missing values
            non_missing_count = total_rows - missing_count
            non_missing_pct = (non_missing_count / total_rows) * 100
            
            print(f"  Unique entries:        {unique_count:,} ({unique_pct:.2f}% of total rows)")
            print(f"  Missing values:        {missing_count:,} ({missing_pct:.2f}%)")
            print(f"  Non-missing values:    {non_missing_count:,} ({non_missing_pct:.2f}%)")
            print()
        
        print("=" * 80)
        
        # Print 20 random rows (only the specified columns)
        print("\n📋 20 RANDOM SAMPLE ROWS")
        print("-" * 80)
        
        # Get 20 random indices
        sample_size = min(20, total_rows)  # In case dataset has fewer than 20 rows
        random_indices = random.sample(range(total_rows), sample_size)
        
        # Display the sampled rows
        sample_df = df.loc[random_indices, columns_to_analyze].reset_index(drop=True)
        
        # Print with nice formatting
        for idx, row in sample_df.iterrows():
            print(f"\nRow {idx + 1}:")
            for col in columns_to_analyze:
                value = row[col]
                # Handle NaN values
                display_value = 'N/A' if pd.isna(value) else value
                print(f"  {col:20s}: {display_value}")
        
        print("\n" + "=" * 80)
        print("✅ Analysis complete!")

except FileNotFoundError:
    print(f"❌ Error: File not found at '{file_path}'")
    print("Please check the file path and ensure the file exists.")
except Exception as e:
    print(f"❌ Error occurred: {type(e).__name__}")
    print(f"Details: {e}")

Loading data from 'data_processed/BRON_cleaned/ongevallen_2024_clean.csv'...

Total rows in dataset: 126,380


📊 COLUMN: junctie_id
--------------------------------------------------------------------------------
  Unique entries:        28,865 (22.84% of total rows)
  Missing values:        92,034 (72.82%)
  Non-missing values:    34,346 (27.18%)


📊 COLUMN: wegvak_id
--------------------------------------------------------------------------------
  Unique entries:        37,632 (29.78% of total rows)
  Missing values:        78,221 (61.89%)
  Non-missing values:    48,159 (38.11%)


📊 COLUMN: straatnaam
--------------------------------------------------------------------------------
  Unique entries:        26,440 (20.92% of total rows)
  Missing values:        0 (0.00%)
  Non-missing values:    126,380 (100.00%)


📋 20 RANDOM SAMPLE ROWS
--------------------------------------------------------------------------------

Row 1:
  junctie_id          : N/A
  wegvak_id           : 280273

C:\Users\tomma\AppData\Local\Temp\ipykernel_15452\3828063260.py:11: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [5]:
import pandas as pd
import json
from collections import defaultdict

# --- Setup ---
csv_file = 'data_processed/BRON_cleaned/ongevallen_2022_clean.csv'
geojson_file = 'export2.geojson'

print("Loading data files...\n")

try:
    # Load CSV file
    df = pd.read_csv(csv_file)
    print(f"✅ Loaded CSV: {len(df):,} rows")
    
    # Load GeoJSON file
    with open(geojson_file, 'r', encoding='utf-8') as f:
        geojson_data = json.load(f)
    
    features = geojson_data.get('features', [])
    print(f"✅ Loaded GeoJSON: {len(features):,} features\n")
    
    # --- Extract street names from CSV ---
    print("=" * 80)
    print("EXTRACTING STREET NAMES FROM CSV")
    print("=" * 80)
    
    csv_streets = df['straatnaam'].dropna().unique()
    csv_streets_set = set(csv_streets)
    print(f"Found {len(csv_streets_set):,} unique street names in CSV\n")
    
    # --- Extract street names and highway types from GeoJSON ---
    print("=" * 80)
    print("EXTRACTING STREET NAMES FROM GEOJSON")
    print("=" * 80)
    
    geojson_streets = set()
    street_to_features = defaultdict(list)  # Map street name to feature IDs
    street_to_highway_types = defaultdict(set)  # Map street name to highway types
    highway_type_counts = defaultdict(int)  # Count features per highway type
    
    for feature in features:
        properties = feature.get('properties', {})
        street_name = properties.get('name')
        highway_type = properties.get('highway', 'unknown')
        
        if street_name:
            geojson_streets.add(street_name)
            feature_id = feature.get('id', 'Unknown')
            street_to_features[street_name].append(feature_id)
            street_to_highway_types[street_name].add(highway_type)
        
        highway_type_counts[highway_type] += 1
    
    print(f"Found {len(geojson_streets):,} unique street names in GeoJSON\n")
    
    # --- Cross-reference analysis ---
    print("=" * 80)
    print("CROSS-REFERENCE ANALYSIS")
    print("=" * 80)
    
    # Find matches
    matching_streets = csv_streets_set.intersection(geojson_streets)
    
    # Find streets only in CSV
    only_in_csv = csv_streets_set - geojson_streets
    
    # Find streets only in GeoJSON
    only_in_geojson = geojson_streets - csv_streets_set
    
    print(f"\n📊 Overall Summary:")
    print(f"  Streets in CSV:              {len(csv_streets_set):,}")
    print(f"  Streets in GeoJSON:          {len(geojson_streets):,}")
    print(f"  ✅ Matching streets:         {len(matching_streets):,}")
    print(f"     → % of CSV streets:       {len(matching_streets)/len(csv_streets_set)*100:.1f}%")
    print(f"     → % of GeoJSON streets:   {len(matching_streets)/len(geojson_streets)*100:.1f}%")
    print(f"  ⚠️  Only in CSV:             {len(only_in_csv):,} ({len(only_in_csv)/len(csv_streets_set)*100:.1f}%)")
    print(f"  ℹ️  Only in GeoJSON:         {len(only_in_geojson):,} ({len(only_in_geojson)/len(geojson_streets)*100:.1f}%)")
    
    # --- Highway type breakdown ---
    print("\n" + "=" * 80)
    print("HIGHWAY TYPE BREAKDOWN IN GEOJSON")
    print("=" * 80)
    
    # Calculate matches per highway type
    highway_type_matches = defaultdict(lambda: {'total': 0, 'matched': 0, 'matched_streets': set()})
    
    for street in geojson_streets:
        for highway_type in street_to_highway_types[street]:
            highway_type_matches[highway_type]['total'] += 1
            if street in matching_streets:
                highway_type_matches[highway_type]['matched'] += 1
                highway_type_matches[highway_type]['matched_streets'].add(street)
    
    print(f"\n{'Highway Type':<20} {'Total':<10} {'Matched':<10} {'Match %':<10}")
    print("-" * 80)
    for highway_type in sorted(highway_type_matches.keys()):
        stats = highway_type_matches[highway_type]
        total = stats['total']
        matched = stats['matched']
        match_pct = (matched / total * 100) if total > 0 else 0
        print(f"{highway_type:<20} {total:<10} {matched:<10} {match_pct:>6.1f}%")
    
    # --- Show matching streets with feature counts ---
    if matching_streets:
        print("\n" + "=" * 80)
        print(f"MATCHING STREETS (showing first 20 of {len(matching_streets)})")
        print("=" * 80)
        
        # Sort by number of accidents in CSV
        street_accident_counts = df[df['straatnaam'].isin(matching_streets)]['straatnaam'].value_counts()
        
        for i, (street, accident_count) in enumerate(street_accident_counts.head(20).items()):
            feature_count = len(street_to_features[street])
            highway_types = ', '.join(street_to_highway_types[street])
            print(f"\n{i+1}. {street}")
            print(f"   Accidents in CSV: {accident_count}")
            print(f"   GeoJSON features: {feature_count}")
            print(f"   Highway types: {highway_types}")
            print(f"   Feature IDs: {', '.join(street_to_features[street][:3])}", end='')
            if feature_count > 3:
                print(f" ... (+{feature_count-3} more)")
            else:
                print()
    
    # --- Show some non-matching streets ---
    if only_in_csv:
        print("\n" + "=" * 80)
        print(f"STREETS ONLY IN CSV (showing first 20 of {len(only_in_csv)})")
        print("=" * 80)
        
        # Get accident counts for these streets
        only_csv_accidents = df[df['straatnaam'].isin(only_in_csv)]['straatnaam'].value_counts()
        
        for i, (street, count) in enumerate(only_csv_accidents.head(20).items()):
            print(f"{i+1}. {street} ({count} accident(s))")
    
    if only_in_geojson:
        print("\n" + "=" * 80)
        print(f"STREETS ONLY IN GEOJSON (showing first 20 of {len(only_in_geojson)})")
        print("=" * 80)
        
        sorted_geojson_only = sorted(only_in_geojson)[:20]
        for i, street in enumerate(sorted_geojson_only, 1):
            feature_count = len(street_to_features[street])
            highway_types = ', '.join(street_to_highway_types[street])
            print(f"{i}. {street} ({feature_count} feature(s), types: {highway_types})")
    
    print("\n" + "=" * 80)
    print("✅ ANALYSIS COMPLETE")
    print("=" * 80)

except FileNotFoundError as e:
    print(f"❌ Error: File not found - {e}")
except Exception as e:
    print(f"❌ Error occurred: {type(e).__name__}")
    print(f"Details: {e}")

Loading data files...

✅ Loaded CSV: 122,036 rows
✅ Loaded GeoJSON: 12,050 features

EXTRACTING STREET NAMES FROM CSV
Found 26,938 unique street names in CSV

EXTRACTING STREET NAMES FROM GEOJSON
Found 2,486 unique street names in GeoJSON

CROSS-REFERENCE ANALYSIS

📊 Overall Summary:
  Streets in CSV:              26,938
  Streets in GeoJSON:          2,486
  ✅ Matching streets:         1,031
     → % of CSV streets:       3.8%
     → % of GeoJSON streets:   41.5%
  ⚠️  Only in CSV:             25,907 (96.2%)
  ℹ️  Only in GeoJSON:         1,455 (58.5%)

HIGHWAY TYPE BREAKDOWN IN GEOJSON

Highway Type         Total      Matched    Match %   
--------------------------------------------------------------------------------
living_street        139        41           29.5%
motorway             7          5            71.4%
motorway_link        6          5            83.3%
primary              36         34           94.4%
primary_link         5          5           100.0%
residential   

In [6]:
import requests
import json
import time

# 1. Define the API endpoint
# (Note: This is the backend API, not the overpass-turbo.eu website)
overpass_url = "https://overpass-api.de/api/interpreter"

# 2. Define a bounding box for the Netherlands
# (S, W, N, E)
bbox_netherlands = "50.75, 3.2, 53.7, 7.22"

# 3. Define the Overpass QL query
# This is the same query as before, just with the new bounding box
overpass_query = f"""
[out:json][timeout:600];
(
  way["highway"~"motorway|primary|secondary|tertiary|residential|living_street"]
     ({bbox_netherlands});
);
out tags geom;
"""

print("🚀 Sending query to Overpass API for all of the Netherlands...")
print("   This will take a long time and will likely time out...")

try:
    start_time = time.time()
    
    # 4. Make the POST request
    # The query is sent as 'data' in the request body
    response = requests.post(overpass_url, data={'data': overpass_query})
    
    end_time = time.time()
    print(f"   ...Query finished in {end_time - start_time:.2f} seconds.\n")

    # 5. Check the HTTP status code
    if response.status_code == 200:
        print("✅ Success! (This is very surprising)")
        
        try:
            # Try to parse the JSON
            data = response.json()
            print(f"   Successfully received and parsed JSON.")
            print(f"   Found {len(data.get('elements', []))} elements.")
            
            # Optionally, save to a file
            # with open("netherlands_roads.json", "w") as f:
            #     json.dump(data, f)
            # print("   Saved data to 'netherlands_roads.json'")

        except json.JSONDecodeError:
            print("❌ Error: Received a 200 OK response, but the content is not valid JSON.")
            print("   This can happen if the query was too large and was terminated.")
            print("   Response text (first 500 chars):")
            print(response.text[:500])

    elif response.status_code == 429:
        print("❌ Error: Too Many Requests (HTTP 429).")
        print("   You have been rate-limited by the API. Wait a while before trying again.")
    
    elif response.status_code == 504:
        print("❌ Error: Gateway Timeout (HTTP 504).")
        print("   As predicted, the query was too large and the server timed out.")

    else:
        print(f"❌ An error occurred. HTTP Status Code: {response.status_code}")
        print("   Response content:")
        print(response.text)

except requests.exceptions.RequestException as e:
    print(f"❌ A network error occurred (e.g., DNS failure, connection refused).")
    print(f"   Error: {e}")

🚀 Sending query to Overpass API for all of the Netherlands...
   This will take a long time and will likely time out...
   ...Query finished in 106.66 seconds.

✅ Success! (This is very surprising)
   Successfully received and parsed JSON.
   Found 1211342 elements.


In [7]:
with open("netherlands_roads.json", "w") as f:
    json.dump(data, f)
print("   Saved data to 'netherlands_roads.json'")

   Saved data to 'netherlands_roads.json'


In [11]:
import pandas as pd
import json
from collections import defaultdict

# --- Setup ---
csv_file = 'data_processed/BRON_cleaned/ongevallen_2022_clean.csv'
osm_file = 'netherlands_roads.json'

print("Loading data files...\n")

try:
    # Load CSV file
    df = pd.read_csv(csv_file)
    print(f"✅ Loaded CSV: {len(df):,} rows")
    
    # Load OSM JSON file
    with open(osm_file, 'r', encoding='utf-8') as f:
        osm_data = json.load(f)
    
    # OSM Overpass API uses 'elements' instead of 'features'
    elements = osm_data.get('elements', [])
    print(f"✅ Loaded OSM data: {len(elements):,} elements\n")
    
    # --- Extract street names from CSV ---
    print("=" * 80)
    print("EXTRACTING STREET NAMES FROM CSV")
    print("=" * 80)
    
    csv_streets = df['straatnaam'].dropna().unique()
    csv_streets_set = set(csv_streets)
    print(f"Found {len(csv_streets_set):,} unique street names in CSV\n")
    
    # --- Extract street names and highway types from OSM ---
    print("=" * 80)
    print("EXTRACTING STREET NAMES FROM OSM DATA")
    print("=" * 80)
    
    osm_streets = set()
    street_to_features = defaultdict(list)  # Map street name to element IDs
    street_to_highway_types = defaultdict(set)  # Map street name to highway types
    highway_type_counts = defaultdict(int)  # Count elements per highway type
    
    for element in elements:
        # OSM data stores tags directly in 'tags' key
        tags = element.get('tags', {})
        street_name = tags.get('name')
        highway_type = tags.get('highway', 'unknown')
        
        if street_name:
            osm_streets.add(street_name)
            element_id = element.get('id', 'Unknown')
            street_to_features[street_name].append(str(element_id))
            street_to_highway_types[street_name].add(highway_type)
        
        highway_type_counts[highway_type] += 1
    
    print(f"Found {len(osm_streets):,} unique street names in OSM data\n")
    
    # --- Cross-reference analysis ---
    print("=" * 80)
    print("CROSS-REFERENCE ANALYSIS")
    print("=" * 80)
    
    # Find matches
    matching_streets = csv_streets_set.intersection(osm_streets)
    
    # Find streets only in CSV
    only_in_csv = csv_streets_set - osm_streets
    
    # Find streets only in OSM
    only_in_osm = osm_streets - csv_streets_set
    
    print(f"\n📊 Overall Summary:")
    print(f"  Streets in CSV:              {len(csv_streets_set):,}")
    print(f"  Streets in OSM:              {len(osm_streets):,}")
    print(f"  ✅ Matching streets:         {len(matching_streets):,}")
    print(f"     → % of CSV streets:       {len(matching_streets)/len(csv_streets_set)*100:.1f}%")
    print(f"     → % of OSM streets:       {len(matching_streets)/len(osm_streets)*100:.1f}%")
    print(f"  ⚠️  Only in CSV:             {len(only_in_csv):,} ({len(only_in_csv)/len(csv_streets_set)*100:.1f}%)")
    print(f"  ℹ️  Only in OSM:             {len(only_in_osm):,} ({len(only_in_osm)/len(osm_streets)*100:.1f}%)")
    
    # --- Highway type breakdown ---
    print("\n" + "=" * 80)
    print("HIGHWAY TYPE BREAKDOWN IN OSM DATA")
    print("=" * 80)
    
    # Calculate matches per highway type
    highway_type_matches = defaultdict(lambda: {'total': 0, 'matched': 0, 'matched_streets': set()})
    
    for street in osm_streets:
        for highway_type in street_to_highway_types[street]:
            highway_type_matches[highway_type]['total'] += 1
            if street in matching_streets:
                highway_type_matches[highway_type]['matched'] += 1
                highway_type_matches[highway_type]['matched_streets'].add(street)
    
    print(f"\n{'Highway Type':<20} {'Total':<10} {'Matched':<10} {'Match %':<10}")
    print("-" * 80)
    for highway_type in sorted(highway_type_matches.keys()):
        stats = highway_type_matches[highway_type]
        total = stats['total']
        matched = stats['matched']
        match_pct = (matched / total * 100) if total > 0 else 0
        print(f"{highway_type:<20} {total:<10} {matched:<10} {match_pct:>6.1f}%")
    
    # --- Show matching streets with feature counts ---
    if matching_streets:
        print("\n" + "=" * 80)
        print(f"MATCHING STREETS (showing first 20 of {len(matching_streets)})")
        print("=" * 80)
        
        # Sort by number of accidents in CSV
        street_accident_counts = df[df['straatnaam'].isin(matching_streets)]['straatnaam'].value_counts()
        
        for i, (street, accident_count) in enumerate(street_accident_counts.head(20).items()):
            feature_count = len(street_to_features[street])
            highway_types = ', '.join(street_to_highway_types[street])
            print(f"\n{i+1}. {street}")
            print(f"   Accidents in CSV: {accident_count}")
            print(f"   OSM elements: {feature_count}")
            print(f"   Highway types: {highway_types}")
            print(f"   Element IDs: {', '.join(street_to_features[street][:3])}", end='')
            if feature_count > 3:
                print(f" ... (+{feature_count-3} more)")
            else:
                print()
    
    # --- Show some non-matching streets ---
    if only_in_csv:
        print("\n" + "=" * 80)
        print(f"STREETS ONLY IN CSV (showing first 20 of {len(only_in_csv)})")
        print("=" * 80)
        
        # Get accident counts for these streets
        only_csv_accidents = df[df['straatnaam'].isin(only_in_csv)]['straatnaam'].value_counts()
        
        for i, (street, count) in enumerate(only_csv_accidents.head(20).items()):
            print(f"{i+1}. {street} ({count} accident(s))")
    
    if only_in_osm:
        print("\n" + "=" * 80)
        print(f"STREETS ONLY IN OSM (showing first 20 of {len(only_in_osm)})")
        print("=" * 80)
        
        sorted_osm_only = sorted(only_in_osm)[:20]
        for i, street in enumerate(sorted_osm_only, 1):
            feature_count = len(street_to_features[street])
            highway_types = ', '.join(street_to_highway_types[street])
            print(f"{i}. {street} ({feature_count} element(s), types: {highway_types})")
    
    print("\n" + "=" * 80)
    print("✅ ANALYSIS COMPLETE")
    print("=" * 80)

except FileNotFoundError as e:
    print(f"❌ Error: File not found - {e}")
except Exception as e:
    print(f"❌ Error occurred: {type(e).__name__}")
    print(f"Details: {e}")

Loading data files...

✅ Loaded CSV: 122,036 rows
✅ Loaded OSM data: 1,211,342 elements

EXTRACTING STREET NAMES FROM CSV
Found 26,938 unique street names in CSV

EXTRACTING STREET NAMES FROM OSM DATA
Found 184,878 unique street names in OSM data

CROSS-REFERENCE ANALYSIS

📊 Overall Summary:
  Streets in CSV:              26,938
  Streets in OSM:              184,878
  ✅ Matching streets:         20,306
     → % of CSV streets:       75.4%
     → % of OSM streets:       11.0%
  ⚠️  Only in CSV:             6,632 (24.6%)
  ℹ️  Only in OSM:             164,572 (89.0%)

HIGHWAY TYPE BREAKDOWN IN OSM DATA

Highway Type         Total      Matched    Match %   
--------------------------------------------------------------------------------
living_street        22384      2724         12.2%
motorway             95         58           61.1%
motorway_link        139        98           70.5%
primary              3212       1451         45.2%
primary_link         593        291          49.1%


In [2]:
import fiona

gpkg_file = 'OSM_filtered_data.gpkg'

# This is 100% safe and instant
layers = fiona.listlayers(gpkg_file)
print(f"Layers found in the file: {layers}")

Layers found in the file: ['points', 'lines', 'multilinestrings', 'multipolygons', 'other_relations']


In [6]:
import fiona

gpkg_file = 'OSM_data_filtered.gpkg'
layers_to_check = ['points', 'lines'] # Add other layers from your list above

print("--- 📊 File Statistics ---")

for layer_name in layers_to_check:
    try:
        with fiona.open(gpkg_file, layer=layer_name) as layer:
            # Get total number of features (rows)
            feature_count = len(layer)
            
            # Get schema (column names and data types)
            schema_properties = layer.schema['properties'].keys()
            
            print(f"\nLayer: '{layer_name}'")
            print(f"  Feature Count: {feature_count:,}")
            print(f"  Columns (Tags): {list(schema_properties)}")
            
    except Exception as e:
        print(f"\nCould not read layer '{layer_name}': {e}")

print("\n--------------------------")

--- 📊 File Statistics ---

Layer: 'points'
  Feature Count: 342,179
  Columns (Tags): ['osm_id', 'name', 'barrier', 'highway', 'ref', 'address', 'is_in', 'place', 'man_made', 'other_tags']

Layer: 'lines'
  Feature Count: 905,321
  Columns (Tags): ['osm_id', 'name', 'highway', 'waterway', 'aerialway', 'barrier', 'man_made', 'railway', 'z_order', 'other_tags']

--------------------------


In [7]:
import geopandas as gpd

gpkg_file = 'OSM_data_filtered.gpkg'
layer_name = 'lines' # Or 'multilinestrings'

# 1. Define a bounding box (minx, miny, maxx, maxy)
# This box is for Amsterdam Centrum
amsterdam_bbox = (4.870, 52.358, 4.918, 52.382)

# 2. Load *only* the roads inside that box
print("Loading roads for Amsterdam Centrum...")
amsterdam_roads = gpd.read_file(
    gpkg_file,
    layer=layer_name,
    bbox=amsterdam_bbox 
)

# 3. Now you have a small, manageable DataFrame
print(f"Successfully loaded {len(amsterdam_roads)} road segments.")
print(amsterdam_roads.head())

Loading roads for Amsterdam Centrum...
Successfully loaded 2333 road segments.
      osm_id                  name      highway waterway aerialway barrier  \
0    7382122              Overtoom    secondary     None      None    None   
1    7373795              Overtoom    secondary     None      None    None   
2  322721654  Tweede Helmersstraat  residential     None      None    None   
3    7373718  Eerste Helmersstraat  residential     None      None    None   
4    7373679  Jacob van Lennepkade  residential     None      None    None   

  man_made railway  z_order  \
0     None    None        6   
1     None    None        6   
2     None    None        3   
3     None    None        3   
4     None    None        3   

                                          other_tags  \
0  "bicycle"=>"use_sidepath","foot"=>"use_sidepat...   
1  "bicycle"=>"use_sidepath","foot"=>"use_sidepat...   
2  "alt_name"=>"2e Helmersstraat","lane_markings"...   
3  "alt_name"=>"1e Helmersstraat","cyclew

In [8]:
gpkg_file = 'OSM_data_filtered.gpkg'

layers = fiona.listlayers(gpkg_file)
print(f"Layers found in the file: {layers}")    

Layers found in the file: ['points', 'lines', 'multilinestrings', 'multipolygons', 'other_relations']


In [12]:
import fiona
from collections import Counter
import time

# --- Configuration ---
gpkg_file = 'OSM_data_filtered.gpkg' 
# We will FORCE it to use 'lines'
roads_layer_name = 'lines' 
# ---------------------

print(f"Opening '{gpkg_file}' (layer: '{roads_layer_name}')...")

# --- Initialization ---
highway_type_counts = Counter()
tag_key_counts = Counter()
total_roads = 0
start_time = time.time()

print("Starting iteration... This may take a minute or two.")

# --- 2. Ultra-Safe Iteration (Method 2) ---
try:
    with fiona.open(gpkg_file, layer=roads_layer_name) as layer:
        for feature in layer:
            total_roads += 1
            
            properties = feature.get('properties', {})
            
            # --- a) Count Highway Types ---
            highway_type = properties.get('highway')
            if highway_type:
                highway_type_counts[highway_type] += 1
                
            # --- b) Count All Tag Keys ---
            for key in properties.keys():
                tag_key_counts[key] += 1

            if total_roads % 500000 == 0:
                print(f"  ...processed {total_roads:,} roads...")

    end_time = time.time()

    if total_roads == 0:
        print("\n❌ Error: Processed 0 roads. Please check that 'lines' is the correct layer.")
        print("Available layers were:", fiona.listlayers(gpkg_file))
    else:
        print(f"\nIteration complete! Processed {total_roads:,} roads in {end_time - start_time:.2f} seconds.")

        # --- 3. Print the Summary ---
        print("\n================== 📊 HIGHWAY TYPE SUMMARY ==================")
        print(f"{'Highway Type':<20} | {'Count':>12} | {'Percentage':>10}")
        print("-" * 46)
        
        for highway_type, count in highway_type_counts.most_common():
            percentage = (count / total_roads) * 100
            print(f"{highway_type:<20} | {count:>12,} | {percentage:>9.2f}%")

        print("\n================== 📊 TAG KEY SUMMARY ==================")
        print("Shows how many roads have each data tag (e.g., 'maxspeed')")
        print(f"{'Tag Key':<20} | {'Count':>12} | {'Percentage':>10}")
        print("-" * 46)

        for tag_key, count in tag_key_counts.most_common():
            percentage = (count / total_roads) * 100 
            print(f"{tag_key:<20} | {count:>12,} | {percentage:>9.2f}%")
            
        print("\n---------------------------------------------------------")

except Exception as e:
    print(f"\n❌ An error occurred during processing: {e}")
    print("Please check the 'gpkg_file' and 'roads_layer_name'.")

Opening 'OSM_data_filtered.gpkg' (layer: 'lines')...
Starting iteration... This may take a minute or two.
  ...processed 500,000 roads...

Iteration complete! Processed 905,321 roads in 28.01 seconds.

================== 📊 HIGHWAY TYPE SUMMARY ==================
Highway Type         |        Count | Percentage
----------------------------------------------
residential          |      387,505 |     42.80%
unclassified         |      204,053 |     22.54%
tertiary             |      118,052 |     13.04%
secondary            |       73,514 |      8.12%
primary              |       47,790 |      5.28%
living_street        |       30,957 |      3.42%
motorway             |       13,593 |      1.50%
motorway_link        |       13,217 |      1.46%
trunk                |        8,558 |      0.95%
trunk_link           |        3,427 |      0.38%
primary_link         |        2,541 |      0.28%
secondary_link       |        1,182 |      0.13%
tertiary_link        |          320 |      0.04%

===

In [15]:
import fiona
from collections import Counter
import time
import re # Import regex for parsing

# --- Configuration ---
gpkg_file = 'OSM_data_filtered.gpkg' # Your .gpkg file name
roads_layer_name = 'lines'
points_layer_name = 'points'
polygons_layer_name = 'multipolygons' # Where schools likely are

# 1. Attributes we want a deep-dive value analysis on
attributes_to_analyze_values_for = [
    'maxspeed', 'surface', 'oneway', 'lit', 'bridge', 
    'tunnel', 'lanes', 'sidewalk', 'cycleway'
]

# 2. Unhelpful tags to ignore in the summary
tags_to_ignore = [
    'osm_id', 'highway', 'waterway', 'aerialway', 'barrier', 
    'man_made', 'railway', 'z_order', 'other_tags', 'name'
]
# ---------------------

def parse_hstore(hstore_string):
    """
    Parses an 'hstore' string (like "key"=>"val","key2"=>"val2")
    into a Python dictionary.
    """
    if hstore_string is None:
        return {}
    # This regex finds all "key"=>"value" pairs
    try:
        return dict(re.findall(r'"(.*?)"=>"(.*?)"', hstore_string))
    except Exception:
        return {} # Return empty on any parsing error

print(f"Opening '{gpkg_file}' for DEEP analysis...")
print(f"Targeting roads: '{roads_layer_name}'")
print(f"Targeting POIs: '{points_layer_name}' and '{polygons_layer_name}'")

# --- Initialization for Roads ---
highway_type_counts = Counter()
attribute_key_counts = Counter() # Counts non-null keys
total_roads = 0
value_counters = {key: Counter() for key in attributes_to_analyze_values_for}
printed_geom_sample = False

start_time = time.time()
print(f"\n--- Starting Part 1: Road Layer Analysis ({roads_layer_name}) ---")

try:
    with fiona.open(gpkg_file, layer=roads_layer_name) as layer:
        for feature in layer:
            total_roads += 1
            properties = feature.get('properties', {})
            
            # --- THIS IS THE FIX ---
            # 1. Get the 'other_tags' string
            other_tags_string = properties.get('other_tags')
            # 2. Parse it into a dictionary
            parsed_tags = parse_hstore(other_tags_string)
            # 3. Merge it with the main properties
            # (Parsed tags will overwrite if there's a conflict, which is good)
            properties.update(parsed_tags)
            # --- END FIX ---

            # --- Print Geometry Sample (once) ---
            if not printed_geom_sample:
                print("\n  --- GEOMETRY SAMPLE (from first road) ---")
                geom_type = feature['geometry']['type']
                # Get the first coordinate of the road
                first_coord = feature['geometry']['coordinates'][0]
                print(f"  Type: {geom_type}")
                print(f"  First coordinate (Lon, Lat): {first_coord}")
                print("  (This confirms geometries are present)\n")
                printed_geom_sample = True

            # --- a) Count Highway Types ---
            highway_type = properties.get('highway')
            if highway_type:
                highway_type_counts[highway_type] += 1
                
            # --- b) Count *Actual Attributes* ---
            for key, value in properties.items():
                if value is not None:
                    # Count all non-null keys
                    if key not in tags_to_ignore:
                        attribute_key_counts[key] += 1
                    
                    # --- c) Count Values for deep-dive ---
                    if key in value_counters:
                        value_counters[key][str(value)] += 1 # Use str(value) for safety

            if total_roads % 250000 == 0:
                print(f"  ...processed {total_roads:,} roads...")

    end_time = time.time()

    if total_roads == 0:
        print(f"\n❌ Error: Processed 0 roads from '{roads_layer_name}'.")
    else:
        print(f"\nIteration complete! Processed {total_roads:,} roads in {end_time - start_time:.2f} seconds.")

        # --- Print Road Summaries ---
        print("\n================== 📊 HIGHWAY TYPE SUMMARY ==================")
        print(f"{'Highway Type':<20} | {'Count':>12} | {'Percentage':>10}")
        print("-" * 46)
        for ht, count in highway_type_counts.most_common():
            print(f"{ht:<20} | {count:>12,} | {(count / total_roads) * 100:>9.2f}%")

        print("\n=============== 📊 TRUE ATTRIBUTE KEY SUMMARY ===============")
        print("Shows how many roads *actually have* a value for each tag.")
        print(f"{'Attribute Key':<20} | {'Count':>12} | {'% of Roads':>10}")
        print("-" * 48)
        for key, count in attribute_key_counts.most_common(30):
            print(f"{key:<20} | {count:>12,} | {(count / total_roads) * 100:>9.2f}%")
        
        print("\n============= 📊 IN-DEPTH ATTRIBUTE VALUE ANALYSIS =============")
        for key, counter in value_counters.items():
            total_with_tag = sum(counter.values())
            print(f"\n--- Value Analysis for '{key}' ({total_with_tag:,} roads have this tag) ---")
            print(f"  {'Value':<25} | {'Count':>12} | {'% of Total':>10}")
            print("  " + "-" * 51)
            for value, count in counter.most_common(15):
                print(f"  {value:<25} | {count:>12,} | {(count / total_roads) * 100:>9.2f}%")
            if len(counter) > 15:
                print(f"  ... and {len(counter) - 15} other unique values ...")

except Exception as e:
    print(f"\n❌ An error occurred processing '{roads_layer_name}': {e}")


# --- Part 2: Analyze POIs (from BOTH layers) ---
print(f"\n\n--- Starting Part 2: POI Layer Analysis ---")
poi_counts = Counter()
total_pois = 0
start_time = time.time()

# We analyze both layers in one loop
for layer_name in [points_layer_name, polygons_layer_name]:
    print(f"  Scanning layer: '{layer_name}'...")
    try:
        with fiona.open(gpkg_file, layer=layer_name) as layer:
            for feature in layer:
                total_pois += 1
                properties = feature.get('properties', {})
                
                # --- THIS IS THE FIX ---
                # Also parse 'other_tags' for POIs, just in case
                other_tags_string = properties.get('other_tags')
                parsed_tags = parse_hstore(other_tags_string)
                properties.update(parsed_tags)
                # --- END FIX ---

                highway_tag = properties.get('highway')
                amenity_tag = properties.get('amenity')
                
                if highway_tag == 'stop':
                    poi_counts['stop_sign'] += 1
                elif highway_tag == 'traffic_signals':
                    poi_counts['traffic_signals'] += 1
                elif highway_tag == 'crossing':
                    poi_counts['crossing'] += 1
                
                if amenity_tag == 'school':
                    poi_counts['school'] += 1
                elif amenity_tag == 'kindergarten':
                    poi_counts['kindergarten'] += 1
                elif amenity_tag == 'university':
                    poi_counts['university'] += 1

    except Exception as e:
        print(f"  ❌ Error processing '{layer_name}': {e}")

end_time = time.time()

if total_pois == 0:
    print(f"\n❌ Error: Processed 0 POIs from all layers.")
else:
    print(f"\nIteration complete! Processed {total_pois:,} total POIs in {end_time - start_time:.2f} seconds.")
    print("\n============= 📊 COMPLETE POI SUMMARY (All Layers) =============")
    print(f"{'Point Type':<20} | {'Count':>12}")
    print("-" * 35)
    for poi_type, count in poi_counts.most_common():
        print(f"{poi_type:<20} | {count:>12,}")

Opening 'OSM_data_filtered.gpkg' for DEEP analysis...
Targeting roads: 'lines'
Targeting POIs: 'points' and 'multipolygons'

--- Starting Part 1: Road Layer Analysis (lines) ---

  --- GEOMETRY SAMPLE (from first road) ---
  Type: LineString
  First coordinate (Lon, Lat): (5.092915, 52.0871952)
  (This confirms geometries are present)

  ...processed 250,000 roads...
  ...processed 500,000 roads...
  ...processed 750,000 roads...

Iteration complete! Processed 905,321 roads in 62.82 seconds.

================== 📊 HIGHWAY TYPE SUMMARY ==================
Highway Type         |        Count | Percentage
----------------------------------------------
residential          |      387,505 |     42.80%
unclassified         |      204,053 |     22.54%
tertiary             |      118,052 |     13.04%
secondary            |       73,514 |      8.12%
primary              |       47,790 |      5.28%
living_street        |       30,957 |      3.42%
motorway             |       13,593 |      1.50%
mo